# AI-Based Loan Approval Prediction & Financial Risk Analysis System
## Comprehensive Exploratory Data Analysis, Feature Engineering, & Machine Learning Modeling

### Problem Statement:
Loan Approval Prediction is a critical task in the fintech and banking sector. Manual assessment of loan applications is labor-intensive, error-prone, and inconsistent. By leveraging historical credit history, demographic, and financial applicant attributes, we build an intelligent machine learning pipeline capable of automated, fair, and data-driven credit risk assessment.

In [1]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Environment and libraries initialized successfully.')

## 1. Load and Inspect the Dataset

In [2]:
df = pd.read_csv('../dataset/loan_data.csv')
print(f'Dataset Shape: {df.shape}')
df.head()

In [3]:
# Dataset Information and Summary Statistics
print('Dataset Info:')
print(df.info())
print('\nSummary Statistics for Numerical Columns:')
display(df.describe())

In [4]:
# Missing Values Check
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_table = pd.DataFrame({'Missing Values': missing, 'Percentage (%)': missing_pct.round(2)})
print('Missing Values Assessment:')
display(missing_table[missing_table['Missing Values'] > 0])

## 2. Exploratory Data Analysis (EDA)
We examine key relationships between applicant features and the target variable `Loan_Status`.

In [5]:
# Target Distribution: Loan Status
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Loan_Status', data=df, palette=['#10b981', '#ef4444'])
plt.title('Loan Approval Distribution (Target Variable)', fontsize=14, fontweight='bold')
plt.xlabel('Loan Status (Y=Approved, N=Rejected)')
plt.ylabel('Count')
plt.show()

In [6]:
# Credit History vs Loan Approval
plt.figure(figsize=(7, 4))
sns.countplot(x='Credit_History', hue='Loan_Status', data=df, palette=['#ef4444', '#10b981'])
plt.title('Credit History vs Loan Approval Status', fontsize=14, fontweight='bold')
plt.xlabel('Credit History (0.0 = Poor / Default, 1.0 = Clean / Repaid)')
plt.legend(['Rejected', 'Approved'])
plt.show()

In [7]:
# Education and Property Area vs Loan Status
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x='Education', hue='Loan_Status', data=df, palette=['#ef4444', '#10b981'], ax=ax1)
ax1.set_title('Education Level vs Loan Status', fontweight='bold')

sns.countplot(x='Property_Area', hue='Loan_Status', data=df, palette=['#ef4444', '#10b981'], ax=ax2)
ax2.set_title('Property Area vs Loan Status', fontweight='bold')
plt.tight_layout()
plt.show()

In [8]:
# Distribution of Applicant Income and Loan Amount
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['ApplicantIncome'], kde=True, ax=ax1, color='#3b82f6', bins=30)
ax1.set_title('Applicant Income Distribution', fontweight='bold')

sns.histplot(df['LoanAmount'].dropna(), kde=True, ax=ax2, color='#8b5cf6', bins=30)
ax2.set_title('Requested Loan Amount Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Missing Value Imputation
- **Categorical Features**: Imputed using Mode (`Gender`, `Married`, `Dependents`, `Self_Employed`, `Credit_History`).
- **Numerical Features**: Imputed using Median (`LoanAmount`, `Loan_Amount_Term`) to resist outlier distortion.

In [9]:
df_clean = df.copy()
if 'Loan_ID' in df_clean.columns:
    df_clean = df_clean.drop('Loan_ID', axis=1)

# Impute missing values
df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].mode()[0])
df_clean['Married'] = df_clean['Married'].fillna(df_clean['Married'].mode()[0])
df_clean['Dependents'] = df_clean['Dependents'].fillna(df_clean['Dependents'].mode()[0])
df_clean['Self_Employed'] = df_clean['Self_Employed'].fillna(df_clean['Self_Employed'].mode()[0])
df_clean['LoanAmount'] = df_clean['LoanAmount'].fillna(df_clean['LoanAmount'].median())
df_clean['Loan_Amount_Term'] = df_clean['Loan_Amount_Term'].fillna(df_clean['Loan_Amount_Term'].mode()[0])
df_clean['Credit_History'] = df_clean['Credit_History'].fillna(df_clean['Credit_History'].mode()[0])

print('Missing values after imputation:', df_clean.isnull().sum().sum())

## 4. Feature Engineering
In credit risk assessment, total household income and debt servicing ability are strong indicators of creditworthiness:
1. `TotalIncome = ApplicantIncome + CoapplicantIncome`
2. `EMI = (LoanAmount * 1000) / Loan_Amount_Term`
3. `BalanceIncome = TotalIncome - EMI`

In [10]:
df_clean['TotalIncome'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']
df_clean['EMI'] = (df_clean['LoanAmount'] * 1000) / df_clean['Loan_Amount_Term']
df_clean['BalanceIncome'] = df_clean['TotalIncome'] - df_clean['EMI']
df_clean[['ApplicantIncome', 'CoapplicantIncome', 'TotalIncome', 'LoanAmount', 'EMI', 'BalanceIncome']].head()

## 5. Categorical Encoding & Dataset Partitioning

In [11]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
df_encoded = df_clean.copy()
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    print(f'Encoded {col}: {dict(zip(le.classes_, range(len(le.classes_))))}')

X = df_encoded.drop('Loan_Status', axis=1)
y = df_encoded['Loan_Status']
print(f'\nFeature Matrix X: {X.shape}, Target y: {y.shape}')

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 6. Machine Learning Model Training & Comparative Benchmark
We evaluate multiple classification algorithms:
- **Logistic Regression** (baseline linear classifier)
- **Decision Tree** (non-linear tree classifier)
- **Random Forest** (ensemble bagging classifier)
- **Gradient Boosting** (ensemble boosting classifier)

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000, random_state=42), True),
    'Decision Tree': (DecisionTreeClassifier(max_depth=4, random_state=42), False),
    'Random Forest': (RandomForestClassifier(n_estimators=250, max_depth=6, random_state=42), False),
    'Gradient Boosting': (GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42), False)
}

results = []
for name, (clf, use_scaled) in models.items():
    x_tr = X_train_scaled if use_scaled else X_train
    x_te = X_test_scaled if use_scaled else X_test
    clf.fit(x_tr, y_train)
    preds = clf.predict(x_te)
    probs = clf.predict_proba(x_te)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall': round(recall_score(y_test, preds), 4),
        'F1-Score': round(f1_score(y_test, preds), 4),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 4)
    })

benchmark_df = pd.DataFrame(results)
display(benchmark_df.sort_values(by='F1-Score', ascending=False))

## 7. Model Evaluation & Confusion Matrix Analysis
Evaluating the chosen Random Forest Classifier in depth.

In [14]:
best_rf = models['Random Forest'][0]
rf_preds = best_rf.predict(X_test)
cm = confusion_matrix(y_test, rf_preds)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Rejected', 'Approved'], yticklabels=['Rejected', 'Approved'])
plt.title('Random Forest Confusion Matrix', fontweight='bold', fontsize=13)
plt.xlabel('Predicted Label')
plt.ylabel('Actual True Label')
plt.show()

print('Detailed Classification Report:')
print(classification_report(y_test, rf_preds, target_names=['Rejected (0)', 'Approved (1)']))

## 8. Feature Importance & Interpretability

In [15]:
importances = best_rf.feature_importances_
feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='#0284c7').invert_yaxis()
plt.title('Feature Importances for Loan Decisioning', fontsize=14, fontweight='bold')
plt.xlabel('Relative Gini Importance')
plt.show()

## 9. Conclusion & Deployment
The trained Random Forest model achieves high recall and accuracy, with `Credit_History`, `BalanceIncome`, and `TotalIncome` identified as primary drivers. The model and preprocessing components are saved and deployed to a Flask web dashboard for real-time predictions.